[//]: # (cr:doc name='chapter_9_business_alignment' id=1e6802b8)
# Chapter 9: Business Alignment

**Purpose:** Align data exploration with business objectives and constraints.

**Outputs:**
- Business context documentation
- Success metrics definition
- Constraints and requirements

---

[//]: # (cr:doc name='9_1_setup' id=291677a7)
## 9.1 Setup

In [ ]:
# @cr:code name='init_progress' id=7333872d
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous
from customer_retention.analysis.visualization import display_table

accept_workflow_params()
track_and_export_previous("09_business_alignment.ipynb")


from customer_retention.analysis.auto_explorer import ExplorationFindings
from customer_retention.core.compat import native_pd
from customer_retention.core.config.experiments import (
    FINDINGS_DIR,
)

# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


In [ ]:
# @cr:code name='load_findings' id=35b16e01
from customer_retention.analysis.auto_explorer import load_notebook_findings

FINDINGS_PATH, _namespace, _ = load_notebook_findings("09_business_alignment.ipynb")
print(f"Using: {FINDINGS_PATH}")

findings = ExplorationFindings.load(FINDINGS_PATH)
print(f"\nLoaded findings for {findings.column_count} columns")

[//]: # (cr:doc name='9_model_diagnostics' id=32ea2cae)

## Model Diagnostics

Assess model health before business alignment: CV stability, feature consistency, overfitting, leakage, calibration.

In [ ]:
# @cr:code name='load_diagnostics_inputs' id=0cf10f74
import json as _json
from pathlib import Path

_skip_diagnostics = True
_diag_data = None
_models = {}

if _namespace and _namespace.exploration_diagnostics_path.exists():
    _diag_data = _json.loads(_namespace.exploration_diagnostics_path.read_text())
    _skip_diagnostics = False
    print(f"Loaded diagnostics for {len(_diag_data.get('cv_results', {}))} models")

    _meta = _json.loads(_namespace.exploration_metadata_path.read_text())
    _mlflow_run_id = _meta.get('mlflow_run_id')

    try:
        import mlflow
        import mlflow.spark

        from customer_retention.core.config.experiments import get_mlflow_dfs_tmpdir
        from customer_retention.stages.modeling import SparkClassifierWrapper

        _saved_uri = _meta.get('mlflow_tracking_uri')
        if _saved_uri:
            mlflow.set_tracking_uri(_saved_uri)
        else:
            from customer_retention.core.compat import is_databricks
            from customer_retention.core.config.experiments import get_experiments_dir
            if not is_databricks():
                mlflow.set_tracking_uri(f"sqlite:///{get_experiments_dir() / 'mlruns.db'}")

        _parent_run = mlflow.get_run(_mlflow_run_id)
        _client = mlflow.tracking.MlflowClient()
        _dfs_tmp = get_mlflow_dfs_tmpdir()

        _child_runs = []
        _all_experiment_ids = [_parent_run.info.experiment_id]
        for _exp in _client.search_experiments():
            if _exp.experiment_id not in _all_experiment_ids:
                _all_experiment_ids.append(_exp.experiment_id)
        _child_runs = _client.search_runs(
            experiment_ids=_all_experiment_ids,
            filter_string=f"tags.mlflow.parentRunId = '{_mlflow_run_id}'",
        )
        print(f"Found {len(_child_runs)} child runs in MLflow")

        for _cr in _child_runs:
            _model_name = _cr.info.run_name
            _artifact_key = f"model_{_model_name.lower().replace(' ', '_')}"
            _model_uri = f"runs:/{_cr.info.run_id}/{_artifact_key}"
            _flavor_tag = _cr.data.tags.get(f"{_artifact_key}.model_flavor", "sklearn")
            print(f"  Loading {_model_name}: uri={_model_uri} flavor={_flavor_tag}")
            try:
                if _flavor_tag == "spark":
                    _load_kwargs = {"dfs_tmpdir": _dfs_tmp} if _dfs_tmp else {}
                    _pipeline = mlflow.spark.load_model(_model_uri, **_load_kwargs)
                    _wdata = _json.loads(Path(_client.download_artifacts(_cr.info.run_id, f"{_artifact_key}_wrapper_meta.json")).read_text())
                    _models[_model_name] = SparkClassifierWrapper.from_pipeline_model(
                        _pipeline, _wdata['spark_model_class'], _wdata['spark_model_params'],
                        _wdata['feature_names'], _wdata.get('class_weight'),
                    )
                else:
                    _models[_model_name] = mlflow.sklearn.load_model(_model_uri)
                print("    OK")
            except Exception as exc:
                print(f"    FAILED: {exc}")
    except Exception as _e:
        print(f"MLflow model reload failed: {_e}")

    if not _models:
        print("No models loaded from MLflow — skipping model-dependent diagnostics")
        _skip_diagnostics = True
else:
    print("No diagnostics inputs found (run NB08 first)")

In [ ]:
# @cr:code name='load_train_test_data' id=1f8111cc
import numpy as np

from customer_retention.core.compat import collect_for_sklearn

_X_train_diag = None
_X_test_diag = None
_y_train_diag = None
_y_test_diag = None

if not _skip_diagnostics:
    from customer_retention.analysis.auto_explorer.active_dataset_store import require_silver_merged
    from customer_retention.stages.modeling.training_preparator import TrainingPreparator

    _silver = require_silver_merged(_namespace)
    _feature_names = _diag_data['feature_names']
    _target = _meta['target_column']

    _preparator = TrainingPreparator(target_column=_target, feature_columns=_feature_names)
    _prep = _preparator.prepare(_silver)

    _X_train_diag = collect_for_sklearn(_prep.X_train[_feature_names])
    _X_test_diag = collect_for_sklearn(_prep.X_test[_feature_names])
    _y_train_diag = collect_for_sklearn(_prep.y_train)
    _y_test_diag = collect_for_sklearn(_prep.y_test)

    del _silver, _prep, _preparator
    print(f"Train: {len(_X_train_diag)}, Test: {len(_X_test_diag)}, Features: {len(_feature_names)}")

In [ ]:
# @cr:code name='run_diagnostics' id=a2809a2a
from customer_retention.analysis.diagnostics import ModelDiagnosticsReportGenerator

_diagnostics_report = None

if not _skip_diagnostics:
    _generator = ModelDiagnosticsReportGenerator()
    _diagnostics_report = _generator.generate(
        models=_models,
        X_train=_X_train_diag, X_test=_X_test_diag,
        y_train=_y_train_diag, y_test=_y_test_diag,
        cv_results=_diag_data['cv_results'],
        train_metrics=_diag_data['train_metrics'],
        test_metrics=_diag_data['test_metrics'],
        feature_names=_diag_data['feature_names'],
        best_model_name=_diag_data['best_model_name'],
        class_proportion=_diag_data['class_proportion'],
    )
    print(f"Diagnostics complete. Verdict: {_diagnostics_report.verdict.upper()}")

In [ ]:
# @cr:code name='display_cv_consistency' id=5ef7b8f4
if _diagnostics_report:
    from customer_retention.analysis.visualization import display_table

    _cv_rows = []
    for _name, _s in _diagnostics_report.summaries.items():
        _cv_rows.append({
            'Model': _name,
            'CV Mean': f"{_s.cv_analysis.cv_mean:.4f}",
            'CV Std': f"{_s.cv_analysis.cv_std:.4f}",
            'Stable': 'Yes' if _s.cv_analysis.passed else 'No',
            'Best-Worst Gap': f"{_s.cv_analysis.best_worst_gap:.4f}",
            'Outlier Folds': str(_s.cv_analysis.outlier_folds) if _s.cv_analysis.outlier_folds else '-',
        })
    display(Markdown('### Cross-Validation Consistency'))
    display_table(native_pd.DataFrame(_cv_rows))

In [ ]:
# @cr:code name='display_feature_stability' id=bada3fed
if _diagnostics_report:
    display(Markdown('### Feature Importance Stability Across Folds'))
    for _name, _s in _diagnostics_report.summaries.items():
        if _s.feature_stability:
            _fs = _s.feature_stability
            print(f"\n{_name}: overall stability = {_fs.overall_stability:.2f}")
            print(f"  Stable features ({len(_fs.stable_features)}): {', '.join(_fs.stable_features[:10])}")
            if _fs.volatile_features:
                print(f"  Volatile features ({len(_fs.volatile_features)}): {', '.join(_fs.volatile_features[:10])}")
        else:
            print(f"\n{_name}: no fold-level importances available")

    _agreement = _diagnostics_report.cross_model_agreement
    display(Markdown('### Cross-Model Feature Agreement'))
    print(f"Overall agreement score: {_agreement.agreement_score:.2f}")
    if _agreement.consensus_features:
        print(f"Consensus features (all models agree): {', '.join(_agreement.consensus_features[:15])}")
    for _pair, _jac in _agreement.pairwise_jaccard.items():
        print(f"  {_pair}: Jaccard = {_jac:.2f}")

In [ ]:
# @cr:code name='display_leakage_and_overfitting' id=355a258b
if _diagnostics_report:
    display(Markdown('### Leakage & Overfitting Analysis'))

    _leakage = _diagnostics_report.leakage
    _critical_leaks = [c for c in getattr(_leakage, 'checks', []) if getattr(c, 'severity', None).name in ('CRITICAL', 'HIGH')]
    if _critical_leaks:
        print(f"  LEAKAGE WARNINGS ({len(_critical_leaks)})")
        for _c in _critical_leaks[:5]:
            print(f"    [{_c.severity.name}] {_c.check_id}: {_c.feature} — {_c.recommendation}")
    else:
        print('  No critical leakage detected')

    print()
    for _name, _s in _diagnostics_report.summaries.items():
        _gap_checks = [c for c in getattr(_s.overfitting, 'checks', []) if hasattr(c, 'gap')]
        if _gap_checks:
            for _c in _gap_checks:
                print(f"  {_name}: {_c.metric} gap={_c.gap:.4f} [{_c.severity.name}]")
        else:
            print(f"  {_name}: no significant train-test gap")

    if _diagnostics_report.best_model_learning_curve:
        _lc = _diagnostics_report.best_model_learning_curve.learning_curve
        if _lc:
            display(Markdown(f'### Learning Curve ({_diag_data["best_model_name"]})'))
            _lc_df = native_pd.DataFrame(_lc)
            display_table(_lc_df)

In [ ]:
# @cr:code name='display_calibration_and_verdict' id=7014d9fe
if _diagnostics_report:
    display(Markdown('### Calibration'))
    for _name, _s in _diagnostics_report.summaries.items():
        _cal = _s.calibration
        print(f"  {_name}: Brier={_cal.brier_score:.4f}  ECE={_cal.ece:.4f}  MCE={_cal.mce:.4f}  → {_cal.recommendation}")

    display(Markdown('---'))
    _v = _diagnostics_report.verdict.upper()
    _emoji = {'SOLID': 'PASS', 'CAUTION': 'REVIEW', 'OVERFIT': 'FAIL', 'LEAKY': 'FAIL', 'UNSTABLE': 'FAIL'}
    display(Markdown(f'## Model Verdict: **{_v}** ({_emoji.get(_v, "?")})'))

    if _diagnostics_report.critical_issues:
        display(Markdown('### Critical Issues'))
        for _issue in _diagnostics_report.critical_issues:
            print(f"  - {_issue}")

    if _diagnostics_report.recommendations:
        display(Markdown('### Recommendations'))
        for _rec in _diagnostics_report.recommendations:
            print(f"  - {_rec}")

In [ ]:
# @cr:code name='score_best_exploration_model' id=c3a7f1e2
_exploration_holdout_metrics = None

if not _skip_diagnostics and _diag_data and _models:
    _best_name = _diag_data.get('best_model_name')
    _best_model = _models.get(_best_name)

    if _best_model is not None and _X_test_diag is not None:
        from sklearn.metrics import (
            accuracy_score,
            average_precision_score,
            confusion_matrix,
            f1_score,
            precision_score,
            recall_score,
            roc_auc_score,
        )

        if hasattr(_best_model, 'predict_proba'):
            _y_proba_best = _best_model.predict_proba(_X_test_diag)[:, 1]
        else:
            import xgboost as xgb
            _y_proba_best = _best_model.predict(xgb.DMatrix(_X_test_diag, feature_names=_diag_data['feature_names']))
        _y_pred_best = (_y_proba_best >= 0.5).astype(int)

        _cm = confusion_matrix(_y_test_diag, _y_pred_best)
        _exploration_holdout_metrics = {
            'model_name': _best_name,
            'dataset': 'exploration_test',
            'n_samples': int(len(_y_test_diag)),
            'roc_auc': float(roc_auc_score(_y_test_diag, _y_proba_best)) if len(np.unique(_y_test_diag)) > 1 else 0.0,
            'pr_auc': float(average_precision_score(_y_test_diag, _y_proba_best)) if len(np.unique(_y_test_diag)) > 1 else 0.0,
            'f1': float(f1_score(_y_test_diag, _y_pred_best, zero_division=0)),
            'precision': float(precision_score(_y_test_diag, _y_pred_best, zero_division=0)),
            'recall': float(recall_score(_y_test_diag, _y_pred_best, zero_division=0)),
            'accuracy': float(accuracy_score(_y_test_diag, _y_pred_best)),
            'confusion_matrix': {'tn': int(_cm[0,0]), 'fp': int(_cm[0,1]), 'fn': int(_cm[1,0]), 'tp': int(_cm[1,1])},
            'probability_stats': {
                'mean': float(np.mean(_y_proba_best)),
                'std': float(np.std(_y_proba_best)),
                'median': float(np.median(_y_proba_best)),
                'p10': float(np.percentile(_y_proba_best, 10)),
                'p90': float(np.percentile(_y_proba_best, 90)),
            },
        }

        display(Markdown(f'### Best Exploration Model: {_best_name}'))
        print(f'  ROC-AUC:   {_exploration_holdout_metrics["roc_auc"]:.4f}')
        print(f'  PR-AUC:    {_exploration_holdout_metrics["pr_auc"]:.4f}')
        print(f'  F1:        {_exploration_holdout_metrics["f1"]:.4f}')
        print(f'  Precision: {_exploration_holdout_metrics["precision"]:.4f}')
        print(f'  Recall:    {_exploration_holdout_metrics["recall"]:.4f}')
        print(f'  Accuracy:  {_exploration_holdout_metrics["accuracy"]:.4f}')
        print(f'\n  Confusion Matrix:  TN={_cm[0,0]:,}  FP={_cm[0,1]:,}')
        print(f'                     FN={_cm[1,0]:,}  TP={_cm[1,1]:,}')
    else:
        print(f"Could not score best model '{_best_name}' — model not loaded or test data unavailable")


In [ ]:
# @cr:code name='save_diagnostics' id=8e138427
if _diagnostics_report:
    findings.metadata['model_diagnostics'] = ModelDiagnosticsReportGenerator.serialize(_diagnostics_report)
    findings.save(FINDINGS_PATH)
    print('Diagnostics saved to findings metadata')

if _exploration_holdout_metrics and _namespace:
    import json as _json
    _diag_on_disk = _json.loads(_namespace.exploration_diagnostics_path.read_text())
    _diag_on_disk['best_model_holdout_metrics'] = _exploration_holdout_metrics
    _namespace.exploration_diagnostics_path.write_text(_json.dumps(_diag_on_disk, default=str))
    print('Best model holdout metrics saved to exploration_diagnostics.json')

del _X_train_diag, _X_test_diag, _y_train_diag, _y_test_diag, _models
del _diagnostics_report, _diag_data, _exploration_holdout_metrics


[//]: # (cr:doc name='9_diagnostics_to_business_separator' id=af136d24)

---
## Business Alignment

Now that we have assessed the model, align with business objectives.

[//]: # (cr:doc name='9_2_business_context' id=fa308f6e)
## 9.2 Business Context

Define the business context for this project.

In [ ]:
# @cr:user_code name='business_context' id=064d7a18
BUSINESS_CONTEXT = {
    "project_name": "Customer Churn Prediction",
    "business_objective": "Reduce customer churn by 20% through proactive retention campaigns",
    "stakeholders": ["Marketing Team", "Customer Success", "Data Science"],
    "timeline": "Q1 2025",
    "budget_constraints": "$50k for retention campaigns per month"
}

print("Business Context:")
for key, value in BUSINESS_CONTEXT.items():
    print(f"  {key}: {value}")

[//]: # (cr:doc name='9_3_success_metrics' id=a90ebf18)
## 9.3 Success Metrics

In [ ]:
# @cr:user_code name='success_metrics' id=b5d8f638
SUCCESS_METRICS = [
    {
        "Metric": "Model AUC",
        "Target": ">= 0.80",
        "Priority": "High",
        "Rationale": "Need strong discrimination to prioritize high-risk customers"
    },
    {
        "Metric": "Precision at 20%",
        "Target": ">= 0.60",
        "Priority": "High",
        "Rationale": "Limited budget means we can only target top 20% of predictions"
    },
    {
        "Metric": "Churn Rate Reduction",
        "Target": "20%",
        "Priority": "High",
        "Rationale": "Primary business objective"
    },
    {
        "Metric": "Model Latency",
        "Target": "< 100ms",
        "Priority": "Medium",
        "Rationale": "Required for real-time scoring"
    },
    {
        "Metric": "Fairness (Demographic Parity)",
        "Target": "Ratio >= 0.8",
        "Priority": "Medium",
        "Rationale": "Ensure equitable treatment across segments"
    }
]

metrics_df = native_pd.DataFrame(SUCCESS_METRICS)
print("Success Metrics:")
display_table(metrics_df)

[//]: # (cr:doc name='9_4_deployment_requirements' id=169f284c)
## 9.4 Deployment Requirements

In [ ]:
# @cr:user_code name='deployment_requirements' id=11c68fa9
DEPLOYMENT_REQUIREMENTS = {
    "scoring_mode": "Both batch and real-time",
    "batch_frequency": "Daily",
    "real_time_latency": "< 100ms p99",
    "infrastructure": "Databricks",
    "model_registry": "MLflow",
    "monitoring": "Required - drift detection and performance tracking",
    "retraining": "Monthly or on significant drift"
}

print("Deployment Requirements:")
for key, value in DEPLOYMENT_REQUIREMENTS.items():
    print(f"  {key}: {value}")

[//]: # (cr:doc name='9_5_data_constraints' id=12e610b1)
## 9.5 Data Constraints

In [ ]:
# @cr:user_code name='data_constraints' id=24109e0b
DATA_CONSTRAINTS = [
    {
        "Constraint": "PII Handling",
        "Requirement": "No direct PII in features (names, SSN, etc.)",
        "Status": "To verify"
    },
    {
        "Constraint": "Data Freshness",
        "Requirement": "Features must be available within 24 hours",
        "Status": "To verify"
    },
    {
        "Constraint": "Historical Depth",
        "Requirement": "Minimum 12 months of history for training",
        "Status": "To verify"
    },
    {
        "Constraint": "Protected Attributes",
        "Requirement": "Age, gender, race should not be direct features",
        "Status": "To verify"
    }
]

constraints_df = native_pd.DataFrame(DATA_CONSTRAINTS)
print("Data Constraints:")
display_table(constraints_df)

[//]: # (cr:doc name='9_6_intervention_strategy' id=fc454427)
## 9.6 Intervention Strategy

In [ ]:
# @cr:user_code name='interventions' id=67b11574
INTERVENTIONS = [
    {
        "Risk Level": "High (>0.8)",
        "Intervention": "Personal call from account manager",
        "Cost": "$50/customer",
        "Expected Effectiveness": "40% retention"
    },
    {
        "Risk Level": "Medium (0.5-0.8)",
        "Intervention": "Personalized email + discount offer",
        "Cost": "$10/customer",
        "Expected Effectiveness": "20% retention"
    },
    {
        "Risk Level": "Low (<0.5)",
        "Intervention": "Automated engagement email",
        "Cost": "$0.50/customer",
        "Expected Effectiveness": "5% retention"
    }
]

interventions_df = native_pd.DataFrame(INTERVENTIONS)
print("Intervention Strategy:")
display_table(interventions_df)

[//]: # (cr:doc name='9_7_save_business_context_to_findings' id=b7d3a943)
## 9.7 Save Business Context to Findings

In [ ]:
# @cr:code name='save_business_metadata' id=e62fbb91
findings.metadata = findings.metadata or {}
findings.metadata["business_context"] = BUSINESS_CONTEXT
findings.metadata["success_metrics"] = SUCCESS_METRICS
findings.metadata["deployment_requirements"] = DEPLOYMENT_REQUIREMENTS

findings.save(FINDINGS_PATH)
print(f"Business context saved to: {FINDINGS_PATH}")


In [ ]:
# @cr:code name='release_stage_memory' id=d339027c
from customer_retention.core.compat import release_stage_memory

release_stage_memory()

[//]: # (cr:doc name='next_steps' id=e960331a)
---

## Next Steps

Continue to **10_spec_generation.ipynb** to generate production specifications.

[//]: # (cr:doc name='section' id=de37c163)
> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.